# Lineare Regression: Spieldauer vorhersagen

Einfache Regression: Kann man die Spieldauer (Turns) basierend auf dem Elo-Rating vorhersagen?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Lade Daten
df = pd.read_csv('games.csv')

# Berechne durchschnittliches Elo
df['avg_elo'] = (df['white_rating'] + df['black_rating']) / 2

# Entferne fehlende Werte
data = df[['avg_elo', 'turns']].dropna()

print(f"Datensatz: {len(data)} Spiele")
print(f"\nZiel: Vorhersage der Spieldauer (Turns) basierend auf durchschnittlichem Elo-Rating")
print(f"\nVariablen:")
print(f"  X (Prädiktor): avg_elo - Durchschnittliches Elo-Rating")
print(f"  y (Ziel): turns - Anzahl Züge im Spiel")

In [ ]:
# Vorbereitung für Regression
X = data[['avg_elo']].values  # Eingabe: avg_elo
y = data['turns'].values      # Zielwert: turns

# Modell trainieren
model = LinearRegression()
model.fit(X, y)

# Vorhersagen
y_pred = model.predict(X)

# Metriken berechnen
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
mae = mean_absolute_error(y, y_pred)
residuals = y - y_pred

print("\n" + "="*70)
print("REGRESSIONSMODELL")
print("="*70)
print(f"\nModell-Gleichung:")
print(f"  Turns = {model.intercept_:.2f} + {model.coef_[0]:.6f} × avg_elo")

print(f"\nModell-Performance:")
print(f"  R² Score: {r2:.4f}")
print(f"  RMSE: {rmse:.2f}")
print(f"  MAE: {mae:.2f}")

print(f"\nInterpretation:")
print(f"  • Das Modell erklärt {r2*100:.1f}% der Varianz")
print(f"  • Durchschnittlicher Fehler: ±{mae:.1f} Züge")
print(f"  • Pro 100 Elo-Punkte: {model.coef_[0]*100:.4f} Züge {'mehr' if model.coef_[0] > 0 else 'weniger'}")

In [ ]:
# Visualisierung: Scatter Plot + Regressionslinie
fig, ax = plt.subplots(figsize=(12, 7))

# Scatter Plot
ax.scatter(X, y, alpha=0.3, s=20, color='steelblue', label='Beobachtete Daten')

# Regressionslinie
X_line = np.array([[X.min()], [X.max()]])
y_line = model.predict(X_line)
ax.plot(X_line, y_line, color='red', linewidth=3, label=f'Regressionslinie (R² = {r2:.3f})')

ax.set_xlabel('Durchschnittliches Elo-Rating', fontsize=12)
ax.set_ylabel('Spieldauer (Turns)', fontsize=12)
ax.set_title('Lineare Regression: Elo Rating → Spieldauer', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Residual-Analyse
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Residual-Analyse', fontsize=14, fontweight='bold')

# 1. Residuals vs Predicted
axes[0, 0].scatter(y_pred, residuals, alpha=0.3, s=20, color='steelblue')
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Vorhergesagte Werte')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Predicted Values')
axes[0, 0].grid(alpha=0.3)

# 2. Histogram der Residuals
axes[0, 1].hist(residuals, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
axes[0, 1].set_xlabel('Residuals')
axes[0, 1].set_ylabel('Häufigkeit')
axes[0, 1].set_title('Distribution der Residuals')
axes[0, 1].grid(alpha=0.3)

# 3. Q-Q Plot (Normalverteilung)
from scipy import stats
stats.probplot(residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot (Normalverteilungs-Test)')
axes[1, 0].grid(alpha=0.3)

# 4. Actual vs Predicted
axes[1, 1].scatter(y, y_pred, alpha=0.3, s=20, color='steelblue')
axes[1, 1].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=2, label='Perfect Prediction')
axes[1, 1].set_xlabel('Tatsächliche Werte')
axes[1, 1].set_ylabel('Vorhergesagte Werte')
axes[1, 1].set_title('Actual vs Predicted')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Beispiel-Vorhersagen
print("\n" + "="*70)
print("BEISPIEL-VORHERSAGEN")
print("="*70)

test_elos = [1000, 1200, 1400, 1600, 1800, 2000]

print(f"\nWenn das durchschnittliche Elo ist, dann wird die Spieldauer etwa sein:\n")

for elo in test_elos:
    predicted_turns = model.predict([[elo]])[0]
    print(f"  Elo {elo}: ~{predicted_turns:.1f} Züge")

# Visualisierung der Vorhersagen
test_X = np.array(test_elos).reshape(-1, 1)
test_y = model.predict(test_X)

fig, ax = plt.subplots(figsize=(10, 6))

# Ursprüngliche Regressionslinie
X_sorted = np.sort(X.flatten())
ax.scatter(X, y, alpha=0.2, s=10, color='gray', label='Alle Daten')
ax.plot(X_sorted, model.predict(X_sorted.reshape(-1, 1)), 
        color='blue', linewidth=2, label='Regressionslinie')

# Test-Punkte hervorheben
ax.scatter(test_X, test_y, s=150, color='red', marker='*', 
          edgecolors='darkred', linewidth=2, label='Beispiel-Vorhersagen', zorder=5)

ax.set_xlabel('Elo Rating')
ax.set_ylabel('Spieldauer (Turns)')
ax.set_title('Beispiel-Vorhersagen mit dem Modell')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Zusammenfassung
print("\n" + "="*70)
print("ZUSAMMENFASSUNG: LINEARE REGRESSION")
print("="*70)

print(f"\n📊 FORSCHUNGSFRAGE")
print(f"   Kann man die Spieldauer vorhersagen, wenn man das Elo-Rating kennt?")

print(f"\n📈 MODELL")
print(f"   Formel: Turns = {model.intercept_:.2f} + {model.coef_[0]:.6f} × avg_elo")

print(f"\n🎯 MODEL-QUALITÄT")
print(f"   R² = {r2:.4f}  ({r2*100:.1f}% der Varianz erklärt)")
if r2 > 0.5:
    print(f"   → GUTE Modell-Qualität")
elif r2 > 0.3:
    print(f"   → MODERATE Modell-Qualität")
else:
    print(f"   → SCHWACHE Modell-Qualität")

print(f"   RMSE = {rmse:.2f}  (Root Mean Squared Error)")
print(f"   MAE = {mae:.2f}   (Mean Absolute Error)")

print(f"\n💡 ERKENNTNISSE")
if model.coef_[0] > 0:
    print(f"   ✓ POSITIVE Korrelation: Höherere Elos → längere Spiele")
    print(f"   • Pro 100 Elo-Punkte: +{model.coef_[0]*100:.2f} Züge")
else:
    print(f"   ✗ NEGATIVE Korrelation: Höherere Elos → kürzere Spiele")
    print(f"   • Pro 100 Elo-Punkte: {model.coef_[0]*100:.2f} Züge")

print(f"\n✅ RESIDUALS")
print(f"   Mean: {residuals.mean():.4f} (sollte ~0 sein)")
print(f"   Std: {residuals.std():.2f}")
print(f"   Zwischen {residuals.min():.2f} und {residuals.max():.2f}")